# HCR register — exploration scratchpad

Exploration only; **nothing load-bearing lives here**. Every result that
matters is computed by the `hcr` package and written up in `reports/`.
This notebook exists to poke at the cleaned data interactively.

In [1]:
import pandas as pd

from hcr import ingest, clean, profile, validate

ingest.inventory()

,file,format,sheet,n_rows,n_columns,columns,status
0,hcr2016-2021.xlsx,xlsx,HCRs 2016 Final,116,150,"[URN, Reportable?\n(EU / RIDDOR / EU RIDDOR, E...",ok
1,hcr2016-2021.xlsx,xlsx,HCRs 2017 Final,108,153,"[URN, Reportable?\n(EU / RIDDOR / EU RIDDOR, E...",ok
2,hcr2016-2021.xlsx,xlsx,HCRs 2018 Final,101,150,"[URN, Reportable?\n(EU / RIDDOR / EU RIDDOR, E...",ok
3,hcr2016-2021.xlsx,xlsx,HCRs 2019 Final,128,150,"[URN, Reportable?\n(EU / RIDDOR / EU RIDDOR, E...",ok
4,hcr2016-2021.xlsx,xlsx,HCRs 2020 Final,94,150,"[URN, Reportable?\n(EU / RIDDOR / EU RIDDOR, E...",ok
5,hcr2016-2021.xlsx,xlsx,HCRs 2021 Provisional,91,150,"[URN, Reportable?\n(EU / RIDDOR / EU RIDDOR, E...",ok
6,hcr2016-2021.xlsx,xlsx,Summary Figures,33,10,"[Summary Figures from 1992, Unnamed: 1, Unname...",ok
7,hsr1992–2014.xlsx,xlsx,Intoduction,27,2,"[Unnamed: 0, Unnamed: 1]",ok
8,hsr1992–2014.xlsx,xlsx,Pivot,28,7,"[Unnamed: 0, Unnamed: 1, Unnamed: 2, Unnamed: ...",ok
9,hsr1992–2014.xlsx,xlsx,Results,4657,118,"[Unnamed: 0, Unnamed: 1, Unnamed: 2, Unnamed: ...",ok


In [2]:
df = pd.concat(
    [
        clean.clean_records(t["frame"], year=t["years"][0])
        for t in ingest.load_source_tables()
    ],
    ignore_index=True,
)
dfy = profile.with_event_year(df)
dfy.shape

(5278, 25)

In [3]:
validate.run_all(dfy)

,rule,status,n_failing,n_rows,pass_rate
0,hole_size_within_plausible_bounds,ok,2,5278,0.999621
1,severity_in_permitted_set,ok,1,5278,0.999811
2,date_within_reporting_period,ok,1,5278,0.999811
3,cause_category_resolvable,ok,107,5278,0.979727


In [4]:
# the hole-size round-down step
profile.boundary_concentration(dfy, "hole_diameter_mm", 1.0, 2.0)

,group,count,share
0,= 1.0,1043,0.604988
1,"(1.0, 2.0)",441,0.255800
2,= 2.0,240,0.139211


In [5]:
# exact-value distribution: look for the imperial anchors (6.35, 12.7, 25.4, 50.8)
nd = profile.numeric_distribution(dfy, "hole_diameter_mm")
nd.sort_values("count", ascending=False).head(15)

,value,count,share
363,1.00,1043,0.209270
555,2.00,240,0.048154
660,3.00,143,0.028692
207,0.50,133,0.026685
31,0.10,130,0.026083
815,5.00,113,0.022673
1067,12.70,105,0.021067
1011,10.00,86,0.017255
871,6.00,76,0.015249
745,4.00,69,0.013844


In [6]:
# missingness by year — watch quantity_released_kg collapse after 2018
profile.missingness_by_column_by_year(
    dfy[["hole_diameter_mm", "quantity_released_kg", "severity", "year"]], "year"
).round(3)

,hole_diameter_mm,quantity_released_kg,severity
year,,,
1992,0.025,0.000,0.0
1993,0.025,0.000,0.0
1994,0.038,0.000,0.0
1995,0.023,0.000,0.0
1996,0.041,0.000,0.0
1997,0.058,0.000,0.0
1998,0.079,0.000,0.0
1999,0.064,0.000,0.0
2000,0.030,0.000,0.0


In [7]:
# severity drift across the 1999 criteria change
profile.category_drift(dfy, "severity", "year").round(3)

severity,AWAITING CLASSIFICATION,MAJOR,MINOR,NON-PROCESS,SIGNIFICANT
year,,,,,
1992,0.000,0.025,0.175,0.000,0.800
1993,0.000,0.118,0.319,0.000,0.563
1994,0.000,0.059,0.386,0.000,0.555
1995,0.000,0.082,0.255,0.000,0.664
1996,0.000,0.124,0.336,0.000,0.539
1997,0.000,0.027,0.298,0.000,0.676
1998,0.000,0.088,0.364,0.000,0.548
1999,0.000,0.055,0.387,0.000,0.557
2000,0.000,0.034,0.481,0.000,0.485


In [8]:
# the three report figures, rendered from the package
from hcr import plots

plots.hole_size_distribution(dfy)

<Figure size 1800x880 with 1 Axes>

In [9]:
plots.measurement_missingness(dfy)

<Figure size 1800x840 with 1 Axes>

In [10]:
plots.severity_drift(dfy)

<Figure size 1800x840 with 1 Axes>